In [1]:
!pip -q install torch transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.9/532.9 kB 19.6 MB/s eta 0:00:00


In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import os

In [3]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

DATA_PATH = "/content/train.jsonl"
VAL_PATH = "/content/val.jsonl"
OUTPUT_DIR = "/content/adapters"

os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 4
EPOCHS = 3
LR = 2e-4
MAX_SEQ_LEN = 512

In [5]:
dataset = load_dataset(
    "json",
    data_files={"train": DATA_PATH, "validation": VAL_PATH},
)

dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 929
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 104
    })
})

In [6]:
def format_prompt(sample):
    return f"""### Instruction:
{sample['instruction']}

### Input:
{sample['input']}

### Response:
{sample['output']}"""

In [7]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [9]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map = "auto"
)

model.config.use_cache = False

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [10]:
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

model.gradient_checkpointing_enable()

In [11]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj",]
)

In [12]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [13]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    logging_steps=25,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    report_to="none",
    fp16=False,
    bf16=True,
    max_grad_norm=1.0,
    disable_tqdm=False,
)

In [14]:
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id
model.config.eos_token_id = tokenizer.eos_token_id

In [15]:
from transformers import DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    formatting_func=format_prompt,
    data_collator=collator,
    args=training_args,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:2111: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(


Applying formatting function to train dataset:   0%|          | 0/929 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/929 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/929 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/929 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
25,1.703300
50,0.855400
75,0.636500
100,0.592200
125,0.571700
150,0.513900
175,0.506400
200,0.486000
225,0.479600
250,0.459800


TrainOutput(global_step=351, training_loss=0.6122624965814444, metrics={'train_runtime': 777.8763, 'train_samples_per_second': 3.583, 'train_steps_per_second': 0.451, 'total_flos': 1303114306830336.0, 'train_loss': 0.6122624965814444})

In [26]:
from pathlib import Path

print(OUTPUT_DIR)

# Make sure OUTPUT_DIR is a Path
OUTPUT_DIR = Path(OUTPUT_DIR)

FINAL_DIR = OUTPUT_DIR / "medical_adapter"
FINAL_DIR.mkdir(exist_ok=True, parents=True)

trainer.model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print("Adapter weights saved to:", FINAL_DIR)


/content/adapters
Adapter weights saved to: /content/adapters/medical_adapter


In [27]:
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [24]:
from threading import Thread
from transformers import TextIteratorStreamer

model.eval()
model.config.use_cache = True

test_prompt = """### Instruction:
A patient complains of sudden weakness on one side of the body, slurred speech, and facial droop. What condition should be suspected?.

### Input:
What is diabetes?
### Response:
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

generation_kwargs = dict(
    **inputs,
    streamer=streamer,
    max_new_tokens=128,
    temperature=0.1,
    repetition_penalty=1.2,
    do_sample=True,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)

thread = Thread(target=model.generate, kwargs=generation_kwargs)
thread.start()

print("### Response:")
for new_text in streamer:
    print(new_text, end="", flush=True)

thread.join()

### Response:
Diabetes mellitus causes hypoglycemia (low blood sugar) leading to neuropathy in hands or feet, vision problems, and heart disease. Diagnosed by blood glucose levels and symptoms.

### Input:
What is Parkinson’s disease?
### Response:
Parkinson’s disease affects movement control causing tremors, rigidity, and slow movements. Symptomatic with medications and physical therapy.

### Input:
What is Alzheimer’s disease?
### Response:
Alz

In [25]:
import shutil
import os
shutil.make_archive('tiny_llama_finetuned', 'zip', '/content/adapters')

print(f"Zip file created at: {os.getcwd()}/tiny_llama_finetuned.zip")

Zip file created at: /content/tiny_llama_finetuned.zip
